In [ ]:
# Cell	Purpose
# 1	Install dependencies
# 2	Load model
# 3	Test generation
# 4	Load SAE
# 5	Extract activations
# 6	Top features for last token
# 7	Steering demo
# 8	Compare features across prompts
# 9	Feature identification
# 10	Find Golden Gate Bridge feature
# 11	Steer with Golden Gate Bridge feature

In [ ]:
# [Cell 1] Install dependencies
# === GPT-2 Mechanistic Interpretability ===
# SAE feature extraction + steering on a real pretrained model

!pip install transformer_lens sae_lens transformers torch
!pip install --force-reinstall numpy scipy pandas scikit-learn

# ⚠️ After running this cell, restart the runtime:
# Runtime → Restart runtime (Ctrl+M+.), then run from Cell 2

In [ ]:
# [Cell 2] Load model
import torch
import transformer_lens as tl
import os
from google.colab import drive

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {device}')

drive.mount('/content/drive')
save_path = '/content/drive/MyDrive/gpt2_small_hooked.pt'

if os.path.exists(save_path):
    model = tl.HookedTransformer.from_pretrained('gpt2-small', device='cpu')
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.to(device)
    print("Loaded from Google Drive")
else:
    model = tl.HookedTransformer.from_pretrained('gpt2-small', device=device)
    torch.save(model.state_dict(), save_path)
    print("Downloaded and saved to Google Drive")

print(f'GPT-2 small: {sum(p.numel() for p in model.parameters())/1e6:.0f}M params')
print(f'dim={model.cfg.d_model}, layers={model.cfg.n_layers}, heads={model.cfg.n_heads}')

In [ ]:
# [Cell 3] Test generation
prompt = "First Citizen:\n"
output = model.generate(prompt, max_new_tokens=200, temperature=0.8)
print(output)

## SAE Feature Extraction
Load a pretrained Sparse Autoencoder from SAE Lens and extract features from GPT-2's residual stream.

In [ ]:
# [Cell 4] Load SAE
from sae_lens import SAE

sae, cfg_dict, sparsity = SAE.from_pretrained(
    release="gpt2-small-res-jb",       # Joseph Bloom's GPT-2 SAEs
    sae_id="blocks.8.hook_resid_pre",   # layer 8 residual stream
    device=device
)

print(f"SAE: {sae.cfg.d_in}d input → {sae.cfg.d_sae}d features")
print(f"Expansion factor: {sae.cfg.d_sae // sae.cfg.d_in}x")

In [ ]:
# [Cell 5] Extract activations + SAE features
test_prompt = "The capital of France is"
_, cache = model.run_with_cache(test_prompt)

# Get residual stream activations at layer 8
resid = cache["blocks.8.hook_resid_pre"]  # shape: [batch, seq_len, d_model]
print(f"Residual stream shape: {resid.shape}")

# Encode through SAE → sparse feature activations
feature_acts = sae.encode(resid)  # shape: [batch, seq_len, d_sae]
print(f"Feature activations shape: {feature_acts.shape}")

# How many features fire per token?
tokens = model.to_str_tokens(test_prompt)
for i, tok in enumerate(tokens):
    n_active = (feature_acts[0, i] > 0).sum().item()
    print(f"  Token '{tok}': {n_active} active features out of {sae.cfg.d_sae}")

In [ ]:
# [Cell 6] Top features for last token
last_token_acts = feature_acts[0, -1]  # features for " is"
top_k = 10
top_features = torch.topk(last_token_acts, top_k)

print(f"Top {top_k} features firing on last token '{tokens[-1]}':")
print(f"{'Feature ID':>12} | {'Activation':>10}")
print("-" * 27)
for feat_id, act_val in zip(top_features.indices, top_features.values):
    print(f"{feat_id.item():>12} | {act_val.item():>10.3f}")

## Feature Steering
Inject a specific SAE feature into the residual stream during generation to steer the model's output.

In [ ]:
# [Cell 7] Steering demo
steering_feature_id = top_features.indices[0].item()
steering_vector = sae.W_dec[steering_feature_id]  # decoder direction for this feature
coeff = 10.0  # steering strength (try 5-50)

def steering_hook(resid, hook):
    resid[:, :, :] += coeff * steering_vector
    return resid

# Generate WITHOUT steering
prompt = "I think the best city in the world is"
print("=== Without steering ===")
output_normal = model.generate(prompt, max_new_tokens=50, temperature=0.7)
print(output_normal)

# Generate WITH steering
print(f"\n=== With steering (feature {steering_feature_id}, coeff={coeff}) ===")
with model.hooks(fwd_hooks=[("blocks.8.hook_resid_pre", steering_hook)]):
    output_steered = model.generate(prompt, max_new_tokens=50, temperature=0.7)
print(output_steered)

In [ ]:
# [Cell 8] Compare features across prompts
prompts = [
    "The president of the United States",
    "The Eiffel Tower in Paris",
    "Machine learning algorithms can",
    "The recipe for chocolate cake",
]

for p in prompts:
    _, c = model.run_with_cache(p)
    acts = sae.encode(c["blocks.8.hook_resid_pre"])
    last_acts = acts[0, -1]
    top3 = torch.topk(last_acts, 3)
    toks = model.to_str_tokens(p)
    print(f"'{toks[-1]}' ← \"{p}\"")
    for fid, fval in zip(top3.indices, top3.values):
        print(f"    feature {fid.item():>6}: {fval.item():.3f}")
    print()

## Feature Identification
Run diverse prompts through the model and find which ones maximally activate a given feature — this tells you what concept that feature represents.

In [ ]:
# [Cell 9] Feature identification — what does a feature represent?
FEATURE_ID = 974  # ← Change this to any feature ID you want to investigate

probe_prompts = [
    "The Eiffel Tower in Paris is a famous landmark",
    "London is the capital of England",
    "The president of the United States lives in Washington",
    "Tokyo is a massive city in Japan",
    "Berlin is the capital of Germany",
    "I love eating pizza and pasta in Rome",
    "Sydney Opera House is in Australia",
    "The Great Wall of China is ancient",
    "Moscow is the capital of Russia",
    "Cairo has the pyramids of Egypt",
    "Machine learning models are trained on data",
    "The stock market crashed in 2008",
    "Photosynthesis converts sunlight into energy",
    "Shakespeare wrote plays in the 16th century",
    "The recipe calls for butter and sugar",
    "DNA stores genetic information in cells",
    "The French Revolution started in 1789",
    "Soccer is the most popular sport worldwide",
    "Quantum mechanics describes subatomic particles",
    "The Amazon rainforest produces oxygen",
    "Marie Curie discovered radium in Paris",
    "Bitcoin is a decentralized cryptocurrency",
    "The speed of light is constant",
    "Notre Dame cathedral is in Paris France",
    "Croissants and baguettes are French pastries",
]

results = []
for p in probe_prompts:
    toks = model.to_str_tokens(p)
    _, c = model.run_with_cache(p)
    acts = sae.encode(c["blocks.8.hook_resid_pre"])
    for i, tok in enumerate(toks):
        val = acts[0, i, FEATURE_ID].item()
        if val > 0:
            results.append((val, tok, p))

results.sort(reverse=True)

print(f"Feature {FEATURE_ID} — top activating tokens:\n")
print(f"{'Activation':>10} | {'Token':>12} | Prompt")
print("-" * 70)
for val, tok, prompt in results[:15]:
    print(f"{val:>10.3f} | {tok:>12} | {prompt}")

In [ ]:
# [Cell 11] Steer with Golden Gate Bridge features
# Combine multiple related features for stronger topical steering
gg_features = {
    10261: 30.0,  # "Gate"
    17608: 40.0,  # "Bridge"
    20677: 15.0,  # "Francisco"
    9752:  15.0,  # "Bay"
}

# Build combined steering vector
steering_vector = sum(
    coeff * sae.W_dec[fid] for fid, coeff in gg_features.items()
)

def gg_steering_hook(resid, hook):
    resid[:, :, :] += steering_vector
    return resid

prompt = "I think the best place to visit is"
print("=== Without steering ===")
print(model.generate(prompt, max_new_tokens=50, temperature=0.7))

print(f"\n=== With Golden Gate Bridge steering ===")
with model.hooks(fwd_hooks=[("blocks.8.hook_resid_pre", gg_steering_hook)]):
    print(model.generate(prompt, max_new_tokens=50, temperature=0.7))

In [15]:
# [Cell 11] Steer with Golden Gate Bridge feature
# ← UPDATE gg_feature_id after running Cell 10
gg_feature_id = 0  # ← PUT THE FEATURE ID FROM CELL 10 HERE
coeff = 15.0

steering_vector = sae.W_dec[gg_feature_id]

def gg_steering_hook(resid, hook):
    resid[:, :, :] += coeff * steering_vector
    return resid

prompt = "I think the best place to visit is"
print("=== Without steering ===")
print(model.generate(prompt, max_new_tokens=50, temperature=0.7))

print(f"\n=== With Golden Gate steering (feature {gg_feature_id}, coeff={coeff}) ===")
with model.hooks(fwd_hooks=[("blocks.8.hook_resid_pre", gg_steering_hook)]):
    print(model.generate(prompt, max_new_tokens=50, temperature=0.7))

=== Without steering ===


  0%|          | 0/50 [00:00<?, ?it/s]

I think the best place to visit is online. There is a lot to do in the city, and to be honest, I didn't even know I was going to make it to Tokyo. I think it is basically a walk from Kemon City to Shibuya and back. I

=== With Golden Gate steering (feature 0, coeff=15.0) ===


  0%|          | 0/50 [00:00<?, ?it/s]

I think the best place to visit is the last remaining base of the Italian and Japanese Empires. Its an ancient city of ruins and ruins of temples and cities and I think the best place to visit is the last remaining base of the Italian and Japanese Empires."

The second quarter of the
